## Regression(회귀)

* 회귀: 입력 특성 $x$로부터 **연속적인 값** $y$를 예측하는 지도 학습(supervised learning) 문제
    - 예: 공부 시간 → 시험 점수, 집의 면적 → 집값, 차량 무게 → 연비
* 분류(Classification)와의 차이

| | 회귀 (Regression) | 분류 (Classification) |
|---|---|---|
| 예측 대상 | 연속적인 값 (숫자) | 이산적인 클래스 (범주) |
| 예시 | 점수, 가격, 온도 | 합격/불합격, 스팸/정상, 붓꽃 품종 |
| 대표 모델 | 선형 회귀 | 로지스틱 회귀 |

* 목표: 데이터에 내재된 $x$와 $y$의 관계 $y \approx f(x)$를 학습해서, 새로운 $x$가 들어왔을 때 $y$를 예측

### Linear Regression (선형 회귀)
* 단순 선형 회귀: 하나의 특성(설명 변수(explanatory variable) ```x```)과 연속적인 타깃(응답 변수(response variable) ```y```) 사이의 관계를 모델링.

<center>

$y={w} x + b$

</center>

* $w$: 가중치(weight), 직선의 기울기
* $b$: 편향(bias), 직선의 절편

In [ ]:
from IPython.display import Image
Image(url='https://raw.githubusercontent.com/rickiepark/ml-with-pytorch/main/ch09/figures/09_01.png', width=500)

- 회귀 직선(regression line)
- 오프셋(offset), 잔차(residual): 회귀 직선과 훈련 샘플 사이의 직선 거리

**목표**: 데이터를 가장 잘 설명하는 $w$와 $b$를 찾는 것

#### 1. 데이터 준비: 공부 시간 → 시험 점수

공부 시간(`hours`)으로 시험 점수(`score`)를 예측하는 소규모 데이터를 직접 만들어 봅니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)  # 난수 생성기 (seed를 고정하면 매번 같은 데이터가 생성됨)

hours = np.round(rng.uniform(1, 10, 20), 1)                 # 1~10시간 사이의 공부 시간 20개
score = np.round(5 * hours + 40 + rng.normal(0, 5, 20), 1)  # 실제 관계: 점수 ≈ 5 x 시간 + 40 (+ 노이즈)

df = pd.DataFrame({'hours': hours, 'score': score})
df.head()

In [ ]:
plt.scatter(df['hours'], df['score'])
plt.xlabel('Study hours')
plt.ylabel('Score')
plt.show()

#### 2. 가설 함수 (Hypothesis)

* 입력 $x$가 주어졌을 때 예측값 $\hat{y}$를 계산하는 함수

<center>

$\hat{y} = H(x) = wx + b$

</center>

In [ ]:
x = df['hours'].values
y = df['score'].values

def predict(x, w, b):
    return w * x + b

In [ ]:
# w, b를 여러 가지로 바꿔서 직선을 그려보기
x_line = np.linspace(0, 11, 100)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (w, b) in zip(axes, [(2, 60), (5, 40), (8, 20)]):
    ax.scatter(x, y, c='steelblue', edgecolor='white', s=70)
    ax.plot(x_line, predict(x_line, w, b), color='tomato')
    ax.set_title(f'w={w}, b={b}')
    ax.set_xlabel('Study hours')
axes[0].set_ylabel('Score')
plt.tight_layout()
plt.show()

어떤 직선이 데이터를 가장 잘 설명할까요? 눈으로 보는 대신 **숫자로** 비교할 기준이 필요합니다.

#### 3. 비용 함수 (Cost function)

* 손실 함수(Loss function) = 비용 함수(Cost function): 모델이 예측한 값과 실제 값 간의 차이를 측정하는 함수
* 선형 회귀에서는 일반적으로 평균 제곱 오차 (Mean Squared Error, MSE) 사용

<center>

$L(w,b)= \frac{1}{n}\sum_{i=1}^{n} (y^{(i)}-\hat{y}^{(i)})^2$

</center>

* 오차를 제곱하는 이유: (1) 양수/음수 오차가 서로 상쇄되지 않도록, (2) 큰 오차에 더 큰 벌점을 주기 위해

In [ ]:
def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)

for w, b in [(2, 60), (5, 40), (8, 20)]:
    print(f'w={w}, b={b} → MSE: {mse(y, predict(x, w, b)):.2f}')

##### $w$에 따른 비용의 변화

$b$를 40으로 고정하고 $w$만 바꿔가며 비용을 계산해 봅니다.

In [ ]:
w_list = np.linspace(0, 10, 101)
cost_list = [mse(y, predict(x, w, 40)) for w in w_list]

plt.plot(w_list, cost_list)
plt.xlabel('w')
plt.ylabel('MSE')
plt.title('Cost vs w (b = 40)')
plt.show()

* 비용 곡선은 아래로 볼록한 **2차 함수** 모양
* 곡선의 **가장 낮은 지점**의 $w$가 최적의 가중치
* 이 최저점을 찾아가는 방법 → **4-2. 경사하강법 (Gradient Descent)**

#### 4. 정규 방정식으로 가중치 구하기

단순 선형 회귀는 비용 함수를 미분해서 0이 되는 지점을 수식으로 바로 구할 수 있음

<center>

$\hat{w}= \frac{\sum_{i=1}^n (x_i - \bar{x}) (y_i - \bar{y})}{\sum_{i=1}^n (x_i -\bar{x})^2}$

$\hat{b} = \bar{y}-\hat{w} \bar{x}$

</center>

##### 정규 방정식 유도

비용 함수에 $\hat{y}_i = wx_i + b$를 대입하면

<center>

$L(w,b)= \frac{1}{n}\sum_{i=1}^{n} (y_i - wx_i - b)^2$

</center>

$L$은 $w$, $b$에 대해 아래로 볼록한 2차 함수이므로, **편미분이 0이 되는 지점이 최솟값**

**Step 1. $b$로 편미분**

$$
\frac{\partial L}{\partial b} = -\frac{2}{n}\sum_{i=1}^{n} (y_i - wx_i - b) = 0
$$

$$
\Rightarrow \sum_{i=1}^{n} y_i - w\sum_{i=1}^{n} x_i - nb = 0
\quad\Rightarrow\quad b = \bar{y} - w\bar{x}
$$

* $\bar{x} = \frac{1}{n}\sum x_i$, $\bar{y} = \frac{1}{n}\sum y_i$

**Step 2. $w$로 편미분**

$$
\frac{\partial L}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n} x_i (y_i - wx_i - b) = 0
$$

Step 1의 $b = \bar{y} - w\bar{x}$를 대입하면

$$
\sum_{i=1}^{n} x_i \big( (y_i - \bar{y}) - w(x_i - \bar{x}) \big) = 0
\quad\Rightarrow\quad
w = \frac{\sum_{i=1}^{n} x_i (y_i - \bar{y})}{\sum_{i=1}^{n} x_i (x_i - \bar{x})}
$$

**Step 3. 정리**

분자·분모에 $\bar{x}\sum (x_i - \bar{x})$을 빼도 값이 변하지 않음 $(\sum x_i = n\bar{x})$


$$
\sum_{i=1}^{n} x_i (y_i - \bar{y}) = \sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y}), \qquad
\sum_{i=1}^{n} x_i (x_i - \bar{x}) = \sum_{i=1}^{n} (x_i - \bar{x})^2
$$

$$
\therefore\quad \hat{w}= \frac{\sum_{i=1}^n (x_i - \bar{x}) (y_i - \bar{y})}{\sum_{i=1}^n (x_i -\bar{x})^2}, \qquad \hat{b} = \bar{y}-\hat{w} \bar{x}
$$


In [ ]:
x_mean = np.mean(x)
y_mean = np.mean(y)

w_best = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
b_best = y_mean - w_best * x_mean
print(f'기울기 w: {w_best:.3f}')
print(f'절편 b: {b_best:.3f}')

In [ ]:
plt.scatter(x, y, c='steelblue', edgecolor='white', s=70)
plt.plot(x_line, predict(x_line, w_best, b_best), color='black', lw=2)
plt.xlabel('Study hours')
plt.ylabel('Score')
plt.show()

In [ ]:
# 7시간 공부했을 때의 예상 점수
predict(7, w_best, b_best)

* 특성이 많아지거나 모델이 복잡해지면 이런 공식을 구하기 어려움 → 경사하강법 사용 (4-2)

#### 5. 선형 회귀 모델의 성능 평가

* **MSE** (Mean Squared Error): $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$
* **RMSE** (Root MSE): $\sqrt{MSE}$ → 타깃과 단위가 같아 해석이 쉬움 (예: 평균적으로 몇 점 정도 틀리는지)
* **MAE** (Mean Absolute Error): $\frac{1}{n}\sum |y_i - \hat{y}_i|$
* **$R^2$** (결정 계수, coefficient of determination)

<center>

$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$

</center>

* $R^2$: 타깃의 전체 변동 중 모델이 설명하는 비율
    - 1에 가까울수록 좋음
    - 0이면 항상 평균값($\bar{y}$)으로 예측하는 것과 같은 수준

In [ ]:
def rmse(y, y_pred):
    return np.sqrt(mse(y, y_pred))

def mae(y, y_pred):
    return np.mean(np.abs(y - y_pred))

def r2_score(y, y_pred):
    ss_res = np.sum((y - y_pred) ** 2)       # 잔차 제곱합
    ss_tot = np.sum((y - np.mean(y)) ** 2)   # 전체 제곱합
    return 1 - ss_res / ss_tot

In [ ]:
y_pred = predict(x, w_best, b_best)

print(f'MSE : {mse(y, y_pred):.2f}')
print(f'RMSE: {rmse(y, y_pred):.2f}')
print(f'MAE : {mae(y, y_pred):.2f}')
print(f'R^2 : {r2_score(y, y_pred):.3f}')

In [ ]:
## 실습
# seaborn의 mpg(자동차 연비) 데이터에서 'weight'(차량 무게)로 'mpg'(연비)를 예측하는 단순 선형 회귀 모델을 만드세요.
# 1) 정규 방정식으로 w, b를 구하고 산점도와 회귀 직선을 그리세요.
# 2) MSE, RMSE, MAE, R^2를 계산하세요.

import seaborn as sns
mpg_df = sns.load_dataset('mpg')

In [ ]:
# 데이터 준비
x_car = mpg_df['weight'].values   # 설명 변수: 차량 무게
y_car = mpg_df['mpg'].values      # 타깃: 연비

mpg_df[['weight', 'mpg']].head()

In [ ]:
# 1-1) 정규 방정식으로 w, b 구하기
# TODO: x_car, y_car의 평균 계산
x_car_mean = None
y_car_mean = None

# TODO: 정규 방정식으로 기울기 w와 절편 b 계산
w_car = None
b_car = None

print(f'기울기 w: {w_car}')
print(f'절편 b: {b_car}')

In [ ]:
# 1-2) 산점도와 회귀 직선 그리기
x_car_line = np.linspace(x_car.min(), x_car.max(), 100)

# TODO: 산점도 그리기 (x: weight, y: mpg)

# TODO: 회귀 직선 그리기 (x_car_line, predict 함수 사용)

plt.xlabel('Weight')
plt.ylabel('MPG')
plt.show()

In [ ]:
# 2) 성능 평가
# TODO: 위에서 정의한 predict 함수로 예측값 계산
y_car_pred = None

# TODO: 위에서 정의한 mse, rmse, mae, r2_score 함수로 성능 지표 계산
print(f'MSE : {None}')
print(f'RMSE: {None}')
print(f'MAE : {None}')
print(f'R^2 : {None}')